<a href="https://colab.research.google.com/github/Aljwharah-h/SDAIA-AI-Agents-System-program/blob/main/%D9%90Assignment4_Semantic_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Set up

In [23]:
%pip install langchain-community pypdf


Loading documents

In [24]:
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader

file_path = "https://files.eric.ed.gov/fulltext/ED536788.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

112


In [25]:
print(f"{docs[0].page_content[:200]}\n")
print(docs[0].metadata)

Introduction to  
Data Analysis 
Handbook
Migrant & Seasonal Head Start 
T echnical Assistance Center
Academy for Educational Development
“If 
I knew what 
you were going to use 
the information for  

{'producer': 'Adobe PDF Library 7.0', 'creator': 'Adobe InDesign CS2 (4.0.2)', 'creationdate': '2006-08-07T18:30:15-06:00', 'moddate': '2006-08-07T18:30:39-06:00', 'trapped': '/False', 'source': 'https://files.eric.ed.gov/fulltext/ED536788.pdf', 'total_pages': 112, 'page': 0, 'page_label': 'i'}


Splitting

In [26]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)

print(len(all_splits))

192


Embeddings

In [28]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [29]:
vector_1 = embeddings.embed_query(all_splits[0].page_content)
vector_2 = embeddings.embed_query(all_splits[1].page_content)

assert len(vector_1) == len(vector_2)
print(f"Generated vectors of length {len(vector_1)}\n")
print(vector_1[:10])

Generated vectors of length 768

[-0.06831696629524231, 0.0007002928759902716, -0.04274025186896324, -0.042280178517103195, -0.022585870698094368, 0.05837533622980118, 0.04190812259912491, -0.036462098360061646, 0.048494499176740646, -0.001817591954022646]


Vector stores

In [30]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

In [33]:
ids = vector_store.add_documents(all_splits)
print(ids)

['bc63bc12-788a-4d23-b5d0-6396d9776826', '8b517ae3-8b6e-4846-9ba7-6274bee9f187', 'e4392317-c8b1-42e7-8806-b40274e77e8f', '1bc0932a-8ce0-40c1-9166-24e25bed92f4', 'b4eaef79-ad92-487c-acbf-71633755a8bb', '40d9271a-e259-4ddd-a252-572dd87437a0', 'fbe23700-9511-40a2-a961-806a0af607ad', '5952111f-082d-4f89-9da9-fd8d2431c696', '627b552d-50e7-4b0b-b5fd-04ef983d1028', 'a93ec90a-633e-48d2-a735-e25e98d7e288', 'c47d4143-e5c1-466c-a020-fb7e34c0caf2', '85192a8c-25be-4faa-bc87-48984fcb02bd', 'ba5b9f74-5822-4ff9-9343-537563c7a712', '18758e5e-95b3-41aa-88c9-04cd22a9e355', '1ecd24cf-5df3-41d1-a589-eed1e358d638', 'e31b588d-bbce-479f-81d5-63fe813eaa45', '95d4ba8d-481d-4017-8c3f-1920c1904fff', 'ef387873-d727-4620-a07d-4f4520fa4385', 'babfc4f0-bb95-4d8d-80e8-7426319ef942', 'ff0ad3eb-84dd-45f0-a3cb-caad515e8041', '04d6e040-6884-4b04-ba80-8d29c33e2a90', '2548c275-4d4c-46a8-8efe-4a48792106f0', 'bca42f09-83ac-4ce2-8662-86c57567ab46', '012fadab-c391-49c5-8da3-1d54f55efcf9', 'ad0f482d-a378-4f88-bdfe-041014334ff2',

In [34]:
results = vector_store.similarity_search(
    "What does this document talk about??"
)

print(results[0])

page_content='Sample Planning Table
Purpose Questions
Data 
Collection 
Methods
Needed 
Resources Lead Person Timeframe
Purpose 1
Purpose 2
Purpose 
The planning piece of the framework gives you an opportunity to identify the resources 
that you will need. Some examples of needed resources include the translation of key 
pieces of information for parents, additional clerical support and scheduling meetings in 
conjunction with other activities.
As with other planning pieces within Head Start it is important that you consult with 
the Policy Council (PC) regarding the plan prior to its implementation. Solicit ideas and 
assistance from PC members regarding the proposed data analysis process.
Important Tip: Traditionally many programs find themselves in a crunch when dealing with 
data analysis.  Taking time to plan out your process will save you time in the long run.' metadata={'producer': 'Adobe PDF Library 7.0', 'creator': 'Adobe InDesign CS2 (4.0.2)', 'creationdate': '2006-08-07T18:

In [35]:
# Note that providers implement different scores; the score here
# is a distance metric that varies inversely with similarity.

results = vector_store.similarity_search_with_score("How does the Transformer encoder work?")
doc, score = results[0]
print(f"Score: {score}\n")
print(doc)

Score: 0.19506011590001104

page_content='© AED/TAC-12 Spring 2006. 0' metadata={'producer': 'Adobe PDF Library 7.0', 'creator': 'Adobe InDesign CS2 (4.0.2)', 'creationdate': '2006-08-07T18:30:15-06:00', 'moddate': '2006-08-07T18:30:39-06:00', 'trapped': '/False', 'source': 'https://files.eric.ed.gov/fulltext/ED536788.pdf', 'total_pages': 112, 'page': 95, 'page_label': '90', 'start_index': 0}


In [36]:
embedding = embeddings.embed_query("What are the advantages of the Transformer over RNNs?")

results = vector_store.similarity_search_by_vector(embedding)
print(results[0])

page_content='© AED/TAC-12 Spring 2006. 6
Appendix C: Supplemental                                          Resources' metadata={'producer': 'Adobe PDF Library 7.0', 'creator': 'Adobe InDesign CS2 (4.0.2)', 'creationdate': '2006-08-07T18:30:15-06:00', 'moddate': '2006-08-07T18:30:39-06:00', 'trapped': '/False', 'source': 'https://files.eric.ed.gov/fulltext/ED536788.pdf', 'total_pages': 112, 'page': 101, 'page_label': '96', 'start_index': 0}


Retrievers

In [40]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1},
)

retriever.batch(
    [
        "What is the main architecture proposed in the pdf?",
        "What is data analysis?",

    ],
)

[[Document(id='066afe6c-e0fd-43be-974d-d3361dc5b4e3', metadata={'producer': 'Adobe PDF Library 7.0', 'creator': 'Adobe InDesign CS2 (4.0.2)', 'creationdate': '2006-08-07T18:30:15-06:00', 'moddate': '2006-08-07T18:30:39-06:00', 'trapped': '/False', 'source': 'https://files.eric.ed.gov/fulltext/ED536788.pdf', 'total_pages': 112, 'page': 29, 'page_label': '24', 'start_index': 819}, page_content='Sample Planning Table\nPurpose Questions\nData \nCollection \nMethods\nNeeded \nResources Lead Person Timeframe\nPurpose 1\nPurpose 2\nPurpose \x18\nThe planning piece of the framework gives you an opportunity to identify the resources \nthat you will need. Some examples of needed resources include the translation of key \npieces of information for parents, additional clerical support and scheduling meetings in \nconjunction with other activities.\nAs with other planning pieces within Head Start it is important that you consult with \nthe Policy Council (PC) regarding the plan prior to its impleme